In [1]:
# Processing csv files from the GBD for country specific BMR to xarray format
# Run for each health variable

In [2]:
import xarray as xr
import numpy as np
import pandas as pd

In [3]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"

# === Health variables ===
# COPD (chronic obstructive pulonary disease)
# LRI (lower respiratory infection)
# IHD (ischemic heart disease)
# DM2 (type 2 diabetes)
# LC (tracheal, bronchus, and lung cancer)
# Stroke
health_VAR = "LC"

# Reads the CSV file into a DataFrame
# Chronic Obstructive Pulmonary Disease baseline mortality rate by country
df = pd.read_csv(f"{BMR_DIR}IHME-GBD_2021_DATA-{health_VAR}.csv")

In [4]:
# Options are Number, Percent or Rate (Rate is per 100,000)
df = df[df["metric_name"] == "Rate"]

In [5]:
# Calculate the mean across 1990-2009 for each country
# We use the 1990-2009 mean for the BMR in the year 2000 and in all future
# projections
df_mean = df.groupby("location_name").mean("year").reset_index()

In [6]:
country = df_mean["location_name"]
val = df_mean["val"]  # the mean value [GBD Results Tool User Guide]
upper = df_mean["upper"]  # 95% Confidence Interval Upper Bound
lower = df_mean["lower"]  # 95% Confidence Interval Lower Bound

In [7]:
data = np.stack([lower, val, upper], axis=1)  # shape (204, 3)

# Create the xarray DataArray
da = xr.DataArray(
    data,
    dims=["country", "quantile"],
    coords={
        "country": country,
        "quantile": ["lower", "mean", "upper"]
    },
    name="BMR_by_country"
)

cite = ("Global Burden of Disease Collaborative Network. Global Burden of"
        "Disease Study 2021 (GBD 2021) Results. Seattle, United States: "
        "Institute for Health Metrics and Evaluation (IHME), 2022. Available "
        "from https://vizhub.healthdata.org/gbd-results/.")

da.attrs["description"] = ("Baseline Mortality Rate per country for "
                           f"{health_VAR} from 1990-2009")
da.attrs["citation"] = cite

In [8]:
da.to_netcdf(f"{BMR_DIR}GBD_BMR_Country_{health_VAR}_1990-2009.nc")